# UniDriveVLA — QwenVL3APlanningHead Standalone Export

This notebook exports the **QwenVL3APlanningHead** from the UniDriveVLA model as ONNX and TorchScript.

**Full pipeline exported:**
```
img (B, N_cam, 3, H, W)
  → ResNet-50 backbone        (C5: 2048 channels)
  → FPN neck Conv2d(2048→256) (the key part!)
  → BEV camera-avg + pool     (50×50 grid → 2500 tokens)
  → UnifiedPerceptionDecoder  (Stage 1 perception)
  → VLM stub (Qwen3-VL-2B placeholder, hidden_dim=2048)
  → DirectVLMFusion           (vlm_fusion_cfg=type='direct')
  → UnifiedPerceptionDecoder  (Stage 2 VLM cross-attention)
  → 7 outputs: det_cls, det_bbox, map_cls, map_pts, plan_trajs, plan_scores, vlm_plan
```

**Config source:** `unidrivevla_b2d_stage1_unified_2b_no_cotraining.py`

**Outputs:**
| Name | Shape | Description |
|---|---|---|
| `det_cls` | (B, 900, 10) | Detection class logits (10 nuScenes classes) |
| `det_bbox` | (B, 900, 10) | Box params: cx,cy,cz,w,l,h,sin,cos,vx,vy |
| `map_cls` | (B, 100, 3) | Map class logits: divider/ped_crossing/boundary |
| `map_pts` | (B, 100, 20, 2) | Map polyline BEV waypoints |
| `plan_trajs` | (B, 3, 6, 2) | 3 planning modes × 6 steps × (x,y) |
| `plan_scores` | (B, 3) | Planning mode scores |
| `vlm_plan` | (B, 6, 2) | Direct VLM action head: 6 waypoints |


In [ ]:
# ── Cell 1: Check GPU and runtime ────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU — running on CPU (will be slow, ~2-3 min for export)')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ── Cell 2: Clone the repository ─────────────────────────────────────────────
import os

REPO_URL    = 'https://github.com/ARNiteshKumar/UniDriveVLA_MulticoreWare.git'
BRANCH      = 'claude/nuscenes-mini-dataset-repo-V4wLx'
REPO_DIR    = '/content/UniDriveVLA_MulticoreWare'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !cd {REPO_DIR} && git pull origin {BRANCH}
else:
    print(f'Cloning {REPO_URL} branch {BRANCH}...')
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls scripts/

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Only onnx and onnxruntime needed — torch and torchvision are pre-installed in Colab
!pip install -q onnx onnxruntime

import onnx, onnxruntime
print(f'onnx        : {onnx.__version__}')
print(f'onnxruntime : {onnxruntime.__version__}')

In [ ]:
# ── Cell 4: Run the export ───────────────────────────────────────────────────
#
# Exports:
#   exports/planning_head.onnx       — ONNX model (graph file, ~1 MB)
#   exports/planning_head.onnx.data  — ONNX external weights (~130 MB)
#   exports/planning_head.pt         — TorchScript traced model (~130 MB)
#   exports/planning_head_info.json  — metadata with shapes and stats
#
# Args:
#   --img-h 450   camera image height (nuScenes standard)
#   --img-w 800   camera image width  (nuScenes standard)
#   --num-cams 6  6 surround cameras
#   --bev-h 50    BEV grid 50x50 = 2500 tokens
#   --bev-w 50

!python scripts/export_planning_head.py \
    --output-dir exports/ \
    --img-h 450 \
    --img-w 800 \
    --num-cams 6 \
    --bev-h 50 \
    --bev-w 50

In [ ]:
# ── Cell 5: Show exported files and metadata ──────────────────────────────────
import json
from pathlib import Path

print('=== Exported files ===')
for f in sorted(Path('exports').iterdir()):
    sz = f.stat().st_size / 1e6
    print(f'  {f.name:<35}  {sz:>8.1f} MB')

print()
info = json.loads(Path('exports/planning_head_info.json').read_text())
print('=== planning_head_info.json ===')
print(json.dumps(info, indent=2))

In [ ]:
# ── Cell 6: Inspect ONNX graph — confirm Conv ops (backbone+neck) ─────────────
import onnx
from collections import Counter

model_onnx = onnx.load('exports/planning_head.onnx')
op_counts = Counter(n.op_type for n in model_onnx.graph.node)

print('ONNX graph summary:')
print(f'  Total nodes : {len(model_onnx.graph.node)}')
print(f'  Input name  : {model_onnx.graph.input[0].name}')
print(f'  Input shape : ', end='')
for d in model_onnx.graph.input[0].type.tensor_type.shape.dim:
    print(d.dim_value if d.dim_value else d.dim_param, end=' ')
print()
print()
print('Op type counts (top 15):')
for op, cnt in op_counts.most_common(15):
    marker = ' <-- backbone+neck' if op == 'Conv' else ''
    print(f'  {op:<20}: {cnt:>4}{marker}')

print()
conv_count = op_counts.get('Conv', 0)
if conv_count > 0:
    print(f'[PASS] {conv_count} Conv ops found — backbone and FPN neck are in the ONNX graph')
else:
    print('[FAIL] No Conv ops — backbone/neck missing from export')

In [ ]:
# ── Cell 7: Run ONNX inference and verify outputs ─────────────────────────────
import torch
import numpy as np
import onnxruntime as ort

# Create dummy camera image input — same shape as nuScenes surround cameras
# Shape: (batch=1, N_cam=6, channels=3, height=450, width=800)
dummy_img = torch.zeros(1, 6, 3, 450, 800)

print('Running ONNX inference...')
sess = ort.InferenceSession('exports/planning_head.onnx',
                             providers=['CPUExecutionProvider'])
ort_outs = sess.run(None, {'img': dummy_img.numpy()})

out_names = ['det_cls', 'det_bbox', 'map_cls', 'map_pts',
             'plan_trajs', 'plan_scores', 'vlm_plan']

print()
print('ONNX output shapes:')
for name, out in zip(out_names, ort_outs):
    print(f'  {name:<14}: {list(out.shape)}')

print()
print('Planning trajectories (3 modes, 6 steps, xy):')
print(ort_outs[out_names.index('plan_trajs')].round(4))

print()
print('VLM direct plan (6 waypoints):')
print(ort_outs[out_names.index('vlm_plan')].round(4))

In [ ]:
# ── Cell 8: Load and run TorchScript model ────────────────────────────────────
import torch

print('Loading TorchScript model...')
ts_model = torch.jit.load('exports/planning_head.pt', map_location='cpu')
ts_model.eval()

dummy_img = torch.zeros(1, 6, 3, 450, 800)
print('Running TorchScript inference...')
with torch.no_grad():
    ts_outs = ts_model(dummy_img)

out_names = ['det_cls', 'det_bbox', 'map_cls', 'map_pts',
             'plan_trajs', 'plan_scores', 'vlm_plan']

print()
print('TorchScript output shapes:')
for name, out in zip(out_names, ts_outs):
    print(f'  {name:<14}: {list(out.shape)}')

In [ ]:
# ── Cell 9: Cross-verify ONNX vs TorchScript ─────────────────────────────────
import numpy as np

print('Comparing ONNX vs TorchScript outputs:')
all_ok = True
for name, ts_out, ort_out in zip(out_names, ts_outs, ort_outs):
    diff = abs(ts_out.numpy() - ort_out).max()
    ok = diff < 1e-3
    status = 'OK  ' if ok else 'DIFF'
    print(f'  [{status}] {name:<14} max_diff={diff:.2e}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('All outputs match between ONNX and TorchScript.')
else:
    print('WARNING: some outputs differ between backends.')

In [ ]:
# ── Cell 10: Download exported files ─────────────────────────────────────────
# Downloads the ONNX graph + external weights + info JSON
# Note: planning_head.onnx needs planning_head.onnx.data to run

from google.colab import files
import os

to_download = [
    'exports/planning_head.onnx',
    'exports/planning_head.onnx.data',
    'exports/planning_head.pt',
    'exports/planning_head_info.json',
]

for fp in to_download:
    if os.path.exists(fp):
        sz = os.path.getsize(fp) / 1e6
        print(f'Downloading {fp}  ({sz:.1f} MB)...')
        files.download(fp)
    else:
        print(f'Not found: {fp}')